In [15]:
# Install needed libraries
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score

In [14]:
import lightgbm as lgb

In [5]:
# Compile transcriptome tsvs into one pandas dataframe

# Read the files
df1 = pd.read_csv('Data/TCGA_Transcriptome/TCGA-KICH.star_fpkm-uq.tsv', sep='\t')
df2 = pd.read_csv('Data/TCGA_Transcriptome/TCGA-KIRC.star_fpkm-uq.tsv', sep='\t')
df3 = pd.read_csv('Data/TCGA_Transcriptome/TCGA-KIRP.star_fpkm-uq.tsv', sep='\t')

# Get the name of the first (gene name) column
gene_col = df1.columns[0]

# Set gene name as index for all three
df1 = df1.set_index(gene_col)
df2 = df2.set_index(gene_col)
df3 = df3.set_index(gene_col)

# Concatenate along columns (axis=1)
combined_df = pd.concat([df1, df2, df3], axis=1)

# Reset index to make gene names a column again
combined_df1 = combined_df.reset_index()

# Save to a new TSV
combined_df1.to_csv(r'Data/TCGA_Transcriptome/merged_transcriptome.tsv', sep='\t', index=False)

print(f"\nTranscriptome combined dataframe shape: {combined_df1.shape}")


Transcriptome combined dataframe shape: (60660, 1025)


In [6]:
# Combine clinical data tsvs into one pandas dataframe and drop all unneeded columns

# Read the files
df1 = pd.read_csv('Data/TCGA_Clinical/TCGA-KICH.clinical.tsv', sep='\t')
df2 = pd.read_csv('Data/TCGA_Clinical/TCGA-KIRC.clinical.tsv', sep='\t')
df3 = pd.read_csv('Data/TCGA_Clinical/TCGA-KIRP.clinical.tsv', sep='\t')

# Assign a list of columns we will keep
columns_to_keep = ['sample', 'primary_diagnosis.diagnoses']

# Create new dataframes that ONLY have the columns we want
df1a = df1[columns_to_keep]
df2a = df2[columns_to_keep]
df3a = df3[columns_to_keep]

# Concatenate along rows (axis=0)
combined_df2 = pd.concat([df1a, df2a, df3a], axis=0)

# Save to a new TSV
combined_df2.to_csv(r'Data/TCGA_Clinical/merged_clinical.tsv', sep='\t', index=False)

print(f"\nClinical combined dataframe shape: {combined_df2.shape}")


Clinical combined dataframe shape: (1407, 2)


In [7]:
# Make sure only samples present in BOTH dataframes are included
# Extract names from each dataframe
names_in_columns = set(combined_df1.columns)  # names from df1's column headers
first_col = combined_df2.columns[0]           # get the name of df2's first column
names_in_rows = set(combined_df2[first_col])  # names from df2's first column

# Find names present in BOTH
common_names = names_in_columns & names_in_rows

print(f"Names in df1 (columns): {len(names_in_columns)}")
print(f"Names in df2 (rows):    {len(names_in_rows)}")
print(f"Names in common:        {len(common_names)}")

# Filter df1: keep only columns whose header is a common name
# (preserve any non-name columns like a gene/ID column if needed)
non_name_cols = [combined_df1.columns[0]]  # e.g., keep the first column (gene names, etc.)
name_cols_to_keep = [c for c in combined_df1.columns if c in common_names]
transcriptome = combined_df1[non_name_cols + name_cols_to_keep]
# Transpose df1: make the Subject ID the row names instead, to match df2
transcriptome = transcriptome.set_index(transcriptome.columns[0]) # set Ensembl_ID as index
transcriptome = transcriptome.T # samples as rows, genes as columns
transcriptome.index.name = 'sample' # name the index

# Filter df2: keep only rows where the first column value is a common name
clinical = combined_df2[combined_df2[first_col].isin(common_names)]
clinical.columns = ['sample', 'primary_diagnosis']
clinical = clinical.set_index('sample')

print(f"\nTranscriptome dataframe filtered shape: {transcriptome.shape}")
print(f"Clinical dataframe filtered shape: {clinical.shape}")

transcriptome.to_csv(r'Data/TCGA_Transcriptome/filtered_transcriptome.tsv', sep='\t', index=False)
clinical.to_csv(r'Data/TCGA_Clinical/filtered_clinical.tsv', sep='\t', index=False)

Names in df1 (columns): 1025
Names in df2 (rows):    1407
Names in common:        1024

Transcriptome dataframe filtered shape: (1024, 60660)
Clinical dataframe filtered shape: (1024, 1)


In [8]:
# Variance filtering: only include the genes with the top 5000 variances.
gene_variances = transcriptome.var(axis=0) # variance per gene
top_genes = gene_variances.nlargest(5000).index # top 5000 gene IDs
transcriptome = transcriptome[top_genes] # shape: (1025, 5000)

print(f"X shape after variance filtering: {transcriptome.shape}")


X shape after variance filtering: (1024, 5000)


In [9]:
# Convert target labels to numbers
# Combine labels for all Renal Cell Carcinoma types
clinical['primary_diagnosis'] = clinical['primary_diagnosis'].replace({
    'Renal cell carcinoma, NOS': 'Renal cell carcinoma, NOS or chromophobe type',
    'Renal cell carcinoma, chromophobe type': 'Renal cell carcinoma, NOS or chromophobe type'})

le = LabelEncoder()
clinical['label'] = le.fit_transform(clinical['primary_diagnosis'])

print(f"\nClass mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")


Class mapping:
  0: Clear cell adenocarcinoma, NOS
  1: Papillary adenocarcinoma, NOS
  2: Renal cell carcinoma, NOS or chromophobe type


In [27]:
# Align x and y, then split
# Align on shared samples (same order)
common_samples = transcriptome.index.intersection(clinical.index)
X = transcriptome.loc[common_samples]
y = clinical.loc[common_samples, 'label']

print(f"\nFinal X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"Class distribution:\n{y.value_counts().sort_index()}")

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves subtype proportions in both splits
)

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")


Final X shape: (1024, 5000)
Final y shape: (1024,)
Class distribution:
label
0    596
1    323
2    105
Name: count, dtype: int64

X_train: (819, 5000), X_test: (205, 5000)
y_train: (819,), y_test: (205,)


In [ ]:
# LightGBM
# set up validation set
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train)          # preserves subtype proportions in both splits

# Set up lgb datasets
train_data = lgb.Dataset(X_train2, label=y_train2)
val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

# define parameteres
params = {
    "objective": "multiclass",        # multiclass classification (3+ classes)
    "num_class": 3,                   # 3 classes in dataset
    "metric": "multi_logloss",       # evaluation metric for multiclass -
    "learning_rate": 0.01,        # step size per tree (smaller = slower, more stable learning)
    "num_leaves": 5,             # max leaves per tree (lower = simpler model, less overfitting)
    "min_data_in_leaf": 10,       # minimum samples per leaf (higher = more conservative splits)
    "lambda_l1": 0.1,             # L1 regularization (encourages simpler/sparser model)
    "lambda_l2": 0.1,             # L2 regularization (shrinks weights, improves stability)
    "feature_pre_filter": False, # allows you to change parameters like min_data_in_leaf without errors
}

# Train model
light_mod = lgb.train(
    params,
    train_data,
    num_boost_round=2000, # Max number of trees to build
    valid_sets=[train_data, val_data], # Datasets to evaluate during training
    valid_names=["train", "val"], # Just labels for the outputs
    callbacks=[lgb.early_stopping(stopping_rounds=50)] # Stops training if validation doesn’t improve for 50 rounds
    )

# Predict on test set
pred_proba = light_mod.predict(X_test)
pred = pred_proba.argmax(axis=1)

# Metrics
print(classification_report(y_test, pred, target_names=le.classes_))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.344478 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1129987
[LightGBM] [Info] Number of data points in the train set: 696, number of used features: 5000
[LightGBM] [Info] Start training from score -0.541463
[LightGBM] [Info] Start training from score -1.156278
[LightGBM] [Info] Start training from score -2.268684
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[411]	train's multi_logloss: 0.0257498	val's multi_logloss: 0.257116
                                               precision    recall  f1-score   support

               Clear cell adenocarcinoma, NOS       0.94      0.97      0.95       119
                Papillary adenocarcinoma, NOS       0.95      0.89      0.92        65
Renal cell carcinoma, NOS or chromophobe type       0.85      0.81      0.83        21

                               